In [17]:
import pandas as pd

fre_clu = pd.read_csv('./data/fre_cluster.csv')
age_clu = pd.read_csv('./data/age_cluster.csv')
data = fre_clu[['cno','gender_le','age_grp_le','cluster']]
data = data.rename({'cluster' : 'fre_clu'},axis=1)
data2 =age_clu[['cno','cluster']]
data2 = data2.rename({'cluster' : 'age_clu'},axis=1)
cus = data.merge(data2, on='cno')

In [18]:
cus.head(2)

,cno,gender_le,age_grp_le,fre_clu,age_clu
0,1,1,9,2,2
1,2,1,9,2,2


In [25]:
user_rfm = pd.read_csv('./data/rfm_score.csv', index_col = False)
item_rfm = pd.read_csv('./data/rfm_score1.csv', index_col = False)
user_rfm = user_rfm[['customer','RFM_Score']].rename({'customer':'cno'},axis=1)
item_rfm = item_rfm[['cno','scls_c_nm','RFM_Score']]
item_score = pd.read_csv('./data/rfm_score2.csv', index_col = False)
item_score = item_score[['scls_c_nm','RFM_Score']]

In [53]:
# 데이터 저장
# cus.to_pickle('./data/cus.pkl')
# user_rfm.to_pickle('./data/user_rfm.pkl')
# item_rfm.to_pickle('./data/item_rfm.pkl')
# item_score.to_pickle('./data/item_score.pkl')

In [54]:
# read_data
user_rfm = pd.read_pickle('./data/user_rfm.pkl')
item_rfm = pd.read_pickle('./data/item_rfm.pkl')
cus = pd.read_pickle('./data/cus.pkl')
item_score = pd.read_pickle('./data/item_score.pkl')
user_rfm.head(2), item_rfm.head(2), cus.head(2), item_score.head(2)

(   cno  RFM_Score
 0    1       3.76
 1    2       4.23,
    cno scls_c_nm  RFM_Score
 0    1   Bag&Bag       2.70
 1    1       L.B       1.98,
    cno  gender_le  age_grp_le  fre_clu  age_clu
 0    1          1           9        2        2
 1    2          1           9        2        2,
   scls_c_nm  RFM_Score
 0       14K       3.69
 1    4대 B/D       3.81)

In [10]:
# Join the two dataframes on the scls_c_nm columns
# merged = item_crfm.merge(item_score, left_on = 'scls_c_nm', right_on = 'scls_c_nm', suffixes= ['_user', ''])
# merged.rename(olumns = {'RFM_Score_user':'user_rating'}, inplace = True)
# merged.to_pickle('./data/rating.pkl')


import pandas as pd
import numpy as np
import scipy as sp
from sklearn.metrics.pairwise import cosine_similarity

user_rfm = pd.read_pickle('./data/user_rfm.pkl')
item_rfm = pd.read_pickle('./data/item_rfm.pkl')
cus = pd.read_pickle('./data/cus.pkl')
rating = pd.read_pickle('./data/rating.pkl')

In [11]:
item_rfm.head()

,cno,scls_c_nm,RFM_Score
0,1,Bag&Bag,2.70
1,1,L.B,1.98
2,1,L/C 아웃도어,2.16
3,1,N.B,2.70
4,1,global SPA,2.06


In [8]:
rating[['user_rating']].describe()

,user_rating
count,723242.000000
mean,2.501460
std,1.153976
min,0.300000
25%,1.550000
50%,2.480000
75%,3.420000
max,5.000000


In [197]:
def data_read(num, cus, rating):
    cus_id = int(num)
    id_fre_cluster = cus[cus.cno == cus_id].fre_clu.values[0]
    id_age_cluster = cus[cus.cno == cus_id].age_clu.values[0]
    fre_data = cus[cus.age_clu == id_fre_cluster]
    age_data = cus[cus.age_clu == id_age_cluster]  # .cno.values
    fre_X = fre_data[['cno']].merge(rating, on='cno', how='inner') 
    age_X = age_data[['cno']].merge(rating, on='cno', how='inner')
    return fre_X, age_X

def cal_similarity(df):
    # User CF - row : 사용자 , column : 아이템, values : 평가점수
    # Item CF - row : 아이템 , column : 사용자, values : 평가점수
    piv = df.pivot_table(index=['cno'], columns=['scls_c_nm'], values='user_rating')
    piv_norm = piv.apply(lambda x: (x-np.mean(x))/(np.max(x)-np.min(x)), axis=1) # min-max scaling 
    piv_norm.fillna(0, inplace=True)
    piv_norm = piv_norm.T
    piv_norm = piv_norm.loc[:, (piv_norm != 0).any(axis=0)]
    piv_sparse = sp.sparse.csr_matrix(piv_norm.values)
    item_similarity = cosine_similarity(piv_sparse)
    user_similarity = cosine_similarity(piv_sparse.T)  # 전치
    item_sim_df = pd.DataFrame(item_similarity, index = piv_norm.index, columns = piv_norm.index)
    user_sim_df = pd.DataFrame(user_similarity, index = piv_norm.columns, columns = piv_norm.columns)
    
    return item_sim_df, user_sim_df

# 유사 상품 추천
def top_item(item_nm, item_sim_df):
    li = []
    count = 1
    print('Similar shows to {} include: '.format(item_nm))
    result = item_sim_df.loc[~item_sim_df.index.isin([item_nm]), item_nm].sort_values(ascending = False)[:10]
    for item, score in result.items():
        li.append('No. {}: {}  ({:.2f})'.format(count, item , score))
        count +=1 
    return li

# 유사 유저
def top_users(user, fre_X, user_sim_df):
    li = []
    if user not in fre_X.cno.values:
        li.append('No data available on user {}'.format(user))

    print('Most Similar Users:\n', user)
    result = user_sim_df.sort_values(by=user, ascending=False).loc[:,user][1:11]
    for user, sim in result.items():
        li.append('User #{0}, Similarity value: {1:.2f}'.format(user, sim))
    return li

In [196]:
# 추천 함수
# 입력 : 사용자 / 아이템
# 1 군집 분리
cus_id = int(2)

# 빈도군집, 나이군집
fre_X, age_X = data_read(cus_id, cus, rating)
# 유사도
fre_item_sim_df, fre_user_sim_df = cal_similarity(fre_X)
age_item_sim_df, age_user_sim_df = cal_similarity(age_X)

# 군집의 고객이 가장 많이 구매한 상품 30개중 랜덤 검색하여
random_item = pd.DataFrame(fre_X.scls_c_nm.value_counts()[:30]).sample(n=1).index[0]

print('군집 : ',  cus[cus.cno == cus_id].fre_clu.values[0])
li = top_item(random_item, fre_item_sim_df)
li = top_users(cus_id, fre_X, fre_user_sim_df)

군집 :  2
Similar shows to global SPA include:

Most Similar Users:
 2
